In [38]:
from backcall.backcall import sys
from typing import NewType, List
import numpy as np
import matplotlib.pyplot as plt
import math
import statistics

Point = NewType("Point", tuple[float, float])

LABELS = ["Iris-setosa", "Iris-versicolor", "Iris-virginica"]
FEATURES = ["LS", "CS", "LP", "CP"]


def dist(p: Point, q: Point) -> float:
  return math.sqrt((p[0] - q[0])**2 + (p[1] - q[1])**2)


def normalizacao(x: np.ndarray) -> np.ndarray:
  min = np.min(x)
  max = np.max(x)

  if max == min:
    return np.zeros_like(x, dtype=float)
  return np.array([(xi - min) / (max - min) for xi in x])


def getDadosRotulo(dados: np.ndarray, rotulos: np.ndarray, rotulo: int, indice: int) -> List[int]:
    ret = []
    for idx in range(0, len(dados)):
        if(rotulos[idx] == rotulo):
            ret.append(dados[idx][indice])
    return ret


def visualizaPontos(dados: np.ndarray, rotulos: np.ndarray, d1: int, d2: int):
    fig, ax = plt.subplots()

    ax.scatter(getDadosRotulo(dados, rotulos, 1, d1), getDadosRotulo(dados, rotulos, 1, d2), c='red' , marker='^')
    ax.scatter(getDadosRotulo(dados, rotulos, 2, d1), getDadosRotulo(dados, rotulos, 2, d2), c='blue' , marker='+')
    ax.scatter(getDadosRotulo(dados, rotulos, 3, d1), getDadosRotulo(dados, rotulos, 3, d2), c='green', marker='.')

    plt.show()


def meuKnn(dadosTrain: np.ndarray, rotuloTrain: List[List[int]], dadosTeste: np.ndarray, k: int) -> np.ndarray:
  # Cria matriz de distâncias com dimensão nº de linhas de teste X nº de linhas de treino
  distances = np.ndarray(shape=(dadosTeste.shape[0], dadosTrain.shape[0]))
  predictions = np.ndarray(shape=(dadosTeste.shape[0]))

  for row in range(dadosTeste.shape[0]):
    for col in range(dadosTrain.shape[0]):
      distances[row, col] = dist(dadosTeste[row], dadosTrain[col])

    closest_neighbors = [
        (v, rotuloTrain[i][0])
        for i, v in enumerate(distances[row])
    ]

    closest_neighbors.sort(key=lambda x: x[0])

    # Mantém apenas os K vizinhos mais próximos
    predictions[row] = statistics.mode(
        [rot for (_, rot) in closest_neighbors[:k]]
    )

  return predictions




In [44]:
from scipy.io import loadmat

data = loadmat('grupoDados1.mat')

test_set = normalizacao(data['grupoTest'])
train_set = normalizacao(data['grupoTrain'])
train_labels = data['trainRots']
test_labels = data['testRots']

predicted_labels = meuKnn(train_set, train_labels, test_set, 1)

is_correct = [1 for i in range(len(test_labels)) if test_labels[i] == predicted_labels[i]]

correct_total = sum(is_correct)
total = len(test_labels)

accuracy = correct_total / total

print(f"Acurácia: {accuracy}")

Acurácia: 0.68
